# Here the experimets conducted wiht Gemini 2.5 flash (thinking) is conducted

In [1]:
import os
from pathlib import Path
# os.chdir(Path.cwd().parent)
from data_processing.data_analysis import select_problem_sample_for_model 
gpt5_nano="gemini-2.5-flash (think)"
df = select_problem_sample_for_model(gpt5_nano)
df

,Unnamed: 0,source,problem,competition,unique_problem_label,correct,parsed_answer,gold_answer,output_cost_per_tokens,problem_idx,cost,output_tokens,input_tokens,answer,user_message,idx_answer,model_config,model_name,input_cost_per_tokens,ten_percentile_group
3736,1828,NaN,Suppose $\triangle ABC$ has angles $\angle BAC...,MathArena/aime_2025_outputs,MathArena/aime_2025: 20,False,300,336,3.5,20,0.101926,29094.0,644.0,"Let $A=84^\circ$, $B=60^\circ$, $C=36^\circ$ b...","Please reason step by step, and put your final...",0,gemini/gemini-flash-2.5,gemini-2.5-flash (think),0.15,4
3752,1844,NaN,"Let $ABCDE$ be a convex pentagon with $AB=14$,...",MathArena/aime_2025_outputs,MathArena/aime_2025: 14,False,47,60,3.5,14,0.112694,32191.0,169.0,"Let the vertices be $A, B, C, D, E$. The side ...","Please reason step by step, and put your final...",0,gemini/gemini-flash-2.5,gemini-2.5-flash (think),0.15,1
3780,1872,NaN,Alex divides a disk into four quadrants with t...,MathArena/aime_2025_outputs,MathArena/aime_2025: 13,False,79,204,3.5,13,0.093253,26639.0,109.0,"Let $D$ be the disk. Initially, the disk is di...","Please reason step by step, and put your final...",0,gemini/gemini-flash-2.5,gemini-2.5-flash (think),0.15,2
13604,1856,NaN,"Albert writes $2025$ numbers $a_{1}, \ldots, a...",MathArena/hmmt_feb_2025_outputs,MathArena/hmmt_feb_2025: 18,False,201,\frac{2025}{101},3.5,18,0.152056,43437.0,178.0,Let $N=2025$ and $T=100$. The numbers are $a_1...,"Please reason step by step, and put your final...",0,gemini/gemini-flash-2.5,gemini-2.5-flash (think),0.15,3
18644,776,NaN,Consider a $4 \times 4$ grid of squares. We pl...,MathArena/cmimc_2025_outputs,MathArena/cmimc_2025: 16,False,40,256,3.5,16,0.115904,33112.0,80.0,Let the grid squares be denoted by $c_{ij}$ fo...,"Please reason step by step, and put your final...",0,gemini/gemini-flash-2.5,gemini-2.5-flash (think),0.15,5


In [2]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==5].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

unique_problem_label                             MathArena/cmimc_2025: 16
answer                  Let the grid squares be denoted by $c_{ij}$ fo...
gold_answer                                                           256
ten_percentile_group                                                    5
problem                 Consider a $4 \times 4$ grid of squares. We pl...
Name: 18644, dtype: object

In [ ]:
import textwrap

first_problem_description = df_pruned["problem"]

first_problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=first_problem_description, width=80))
print("answer", first_problem_gold_answer)



Consider a $4 \times 4$ grid of squares. We place coins in some of the grid
squares so that no two coins are orthogonally adjacent, and each $2 \times 2$
square in the grid has at least one coin. How many ways are there to place the
coins?
answer 256


In [ ]:
from multi_agent.multi_agent import Problem
from prompt_template import Reflexion_Solver, Reflector
from reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
problem = problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=first_problem_description,
    answer=first_problem_gold_answer
)

from models.azure_api import Client 
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)